# 🎙️ BookVoice-AI — XTTS-v2 Fine-tuning
**Emotionale Stimme trainieren (Sufi · Rumi · Gedichte)**

---
## ⚠️ Wichtig vor dem Start
1. **Runtime auf Python 3.10 + T4 GPU umstellen:**
   → Laufzeit → Laufzeittyp ändern → Python 3.10 → T4 GPU → Speichern
2. Zellen der Reihe nach ausführen
3. Colab-Tab offen lassen während des Trainings

---
## 🔄 Workflow
1. Python-Version prüfen
2. Server verbinden + Dataset herunterladen
3. Emotionstags hinzufügen (optional)
4. Umgebung installieren (~5-10 Min)
5. GPT Training starten (~2-4 Stunden)
6. DVAE Training (optional, nur bei >20h Audio)
7. Modell exportieren + zu BookVoice hochladen

In [ ]:
# ============================================================
# ZELLE 1: Python-Version prüfen (WICHTIG!)
# ============================================================
import sys

if sys.version_info[:2] != (3, 10):
    print(f"❌ Falsche Python Version: {sys.version_info.major}.{sys.version_info.minor}")
    print()
    print("🔧 Bitte Runtime umstellen:")
    print("   1. Laufzeit → Laufzeittyp ändern")
    print("   2. Python version: 3.10")
    print("   3. Hardware accelerator: T4 GPU")
    print("   4. Speichern → Notebook neu starten")
    raise SystemExit("❌ Gestoppt – bitte auf Python 3.10 umstellen")
else:
    print(f"✅ Python {sys.version_info.major}.{sys.version_info.minor} — passt!")

In [ ]:
# ============================================================
# ZELLE 2: Server & Projekt konfigurieren
# ============================================================
import os, json, requests, zipfile, io
from pathlib import Path

# 🔧 Hier deine Einstellungen eintragen:
SERVER_URL       = "https://ahrar.aksoy-net.de"  #@param {type:"string"}
PROJECT_NAME     = ""                             #@param {type:"string"}
TRAINING_PASSWORD = "sufi2026"                    #@param {type:"string"}
MODELL_NAME      = ""                             # wird automatisch gesetzt

# API-URL normalisieren
if SERVER_URL.rstrip('/').endswith('/api'):
    API_BASE = SERVER_URL.rstrip('/')
else:
    API_BASE = SERVER_URL.rstrip('/') + '/api'

MODELL_NAME = PROJECT_NAME + "_finetuned" if PROJECT_NAME else "finetuned"

print(f"🌐 Server:  {API_BASE}")
print(f"📁 Projekt: {PROJECT_NAME}")
print(f"🤖 Modell:  {MODELL_NAME}")
print()

# Health-Check
print("🔍 Teste Verbindung...")
try:
    r = requests.get(f"{API_BASE}/health", timeout=10)
    if r.status_code == 200:
        print("✅ Server erreichbar")
    else:
        print(f"⚠️ Server Status: {r.status_code}")
except Exception as e:
    print(f"❌ Keine Verbindung: {e}")
    raise

# Auth
print("🔐 Authentifizierung...")
r = requests.post(f"{API_BASE}/training/auth", json={"password": TRAINING_PASSWORD})
if r.status_code == 200:
    print("✅ Trainingsraum-Zugang OK")
else:
    print(f"❌ Falsches Passwort (Status {r.status_code})")
    raise Exception("Authentifizierung fehlgeschlagen")

# Projekt prüfen
print(f"📁 Prüfe Projekt '{PROJECT_NAME}'...")
r = requests.get(f"{API_BASE}/training/projects/{PROJECT_NAME}")
if r.status_code == 200:
    p = r.json()
    print(f"✅ Projekt gefunden — Sprache: {p.get('sprache','?')} · Clips: {p.get('clips',0)}")
    if not p.get('dataset_bereit'):
        print("⚠️ Dataset noch nicht exportiert — bitte im Trainingsraum 'Dataset bauen' klicken")
        raise Exception("Dataset fehlt")
else:
    print(f"❌ Projekt nicht gefunden: {PROJECT_NAME}")
    raise Exception("Projekt fehlt")

In [ ]:
# ============================================================
# ZELLE 3: Dataset herunterladen & entpacken
# ============================================================
import shutil

DATASET_DIR = "/content/dataset"
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

print(f"📥 Lade Dataset herunter...")
dl_url = f"{API_BASE}/training/projects/{PROJECT_NAME}/export/download"
r = requests.get(dl_url, stream=True)
if r.status_code != 200:
    raise Exception(f"Download fehlgeschlagen: {r.status_code}")

total = int(r.headers.get('content-length', 0))
data = b""
for chunk in r.iter_content(8192):
    data += chunk
    if total:
        print(f"\r   {len(data)/total*100:.1f}% ({len(data)//1024} KB)", end="")

print(f"\n✅ Download: {len(data)//1024} KB")

os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as z:
    z.extractall(DATASET_DIR)

# Statistik
lang = Path(f"{DATASET_DIR}/lang.txt").read_text().strip() if Path(f"{DATASET_DIR}/lang.txt").exists() else "tr"
wavs = list(Path(f"{DATASET_DIR}/wavs").glob("*.wav")) if Path(f"{DATASET_DIR}/wavs").exists() else []
train_lines = len(Path(f"{DATASET_DIR}/metadata_train.csv").read_text().splitlines()) - 1 if Path(f"{DATASET_DIR}/metadata_train.csv").exists() else 0
eval_lines  = len(Path(f"{DATASET_DIR}/metadata_eval.csv").read_text().splitlines()) - 1 if Path(f"{DATASET_DIR}/metadata_eval.csv").exists() else 0

print(f"\n📊 Dataset:")
print(f"   Sprache:     {lang}")
print(f"   WAV-Clips:   {len(wavs)}")
print(f"   Train-Paare: {train_lines}")
print(f"   Eval-Paare:  {eval_lines}")

In [ ]:
# ============================================================
# ZELLE 4: Emotionstags hinzufügen (optional)
# ============================================================
ADD_TAGS = True  #@param {type:"boolean"}
# True  = Style-Tags automatisch erkennen (empfohlen für Sufi/Rumi)
# False = Texte ohne Tags trainieren (neutral)

if ADD_TAGS:
    print("🏷️  Füge Style-Tags hinzu...")

    sufi_words     = ['allah', 'aşk', 'gönül', 'can', 'ruh', 'rumi', 'mesnevi', 'sufi', 'derviş', 'hakikat', 'mevlana', 'sevgi', 'nur']
    whisper_words  = ['fısıltı', 'sessizce', 'dua', 'sır', 'gizli', 'kalp', 'huzur']
    dramatic_words = ['yangın', 'fırtına', 'savaş', 'ağla', 'bağır', 'gürledi', 'kahır']

    def auto_tag(text):
        t = text.lower()
        if any(w in t for w in sufi_words):     return f"<sufi>{text}</sufi>"
        if any(w in t for w in whisper_words):  return f"<whisper>{text}</whisper>"
        if any(w in t for w in dramatic_words): return f"<dramatic>{text}</dramatic>"
        if len(text.split()) < 20:              return f"<poem>{text}</poem>"
        return f"<sufi>{text}</sufi>"

    for csv_file in ["metadata_train.csv", "metadata_eval.csv"]:
        p = Path(f"/content/dataset/{csv_file}")
        if not p.exists(): continue
        lines = p.read_text(encoding="utf-8").splitlines()
        new_lines = [lines[0]]
        for line in lines[1:]:
            if not line.strip(): continue
            parts = line.split('|')
            if len(parts) >= 2:
                parts[1] = auto_tag(parts[1].strip())
                new_lines.append('|'.join(parts))
        p.write_text('\n'.join(new_lines), encoding="utf-8")
        print(f"   ✅ {csv_file} getaggt ({len(new_lines)-1} Zeilen)")

    print("✅ Style-Tags fertig")
else:
    print("⏭️ Tags übersprungen — Training mit neutralen Texten")

In [ ]:
# ============================================================
# ZELLE 5: Umgebung installieren (~5-10 Minuten)
# ============================================================
print("⏳ Installiere XTTS Fine-tuning Umgebung...")
print("   Das dauert 5-10 Minuten — bitte warten!")

In [ ]:
%%capture
# Repo klonen
!git clone https://github.com/dikshitrishii/XTTSv2-For-Emotional-Support.git /content/xtts-training
%cd /content/xtts-training

# PyTorch für Python 3.10 + CUDA 11.8
!pip install torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q

# TTS + Trainer
!pip install TTS==0.22.0 -q
!pip install trainer transformers datasets -q

In [ ]:
# Installation prüfen
import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU:     {torch.cuda.get_device_name(0)}")
else:
    print("❌ Keine GPU — bitte Runtime auf T4 umstellen!")
    raise Exception("GPU fehlt")

In [ ]:
# ============================================================
# ZELLE 6: GPT Training starten (~2-4 Stunden)
# ============================================================
print("🚀 Starte GPT Training...")
print()
print("📋 Parameter:")
print("   batch_size:           4")
print("   gradient_accumulation: 4  (effektiv: 16)")
print("   learning_rate:        5e-6")
print("   epochs:               5")
print("   precision:            fp32  (wichtig für XTTS!)")
print()
print("⏱️ Dauer: ca. 2-4 Stunden auf T4")
print("   Colab-Tab offen lassen!")
print()

!python train_gpt.py \
    --dataset_path /content/dataset \
    --output_path /content/output_gpt \
    --mixed_precision no \
    --batch_size 4 \
    --gradient_accumulation 4 \
    --lr 5e-6 \
    --num_epochs 5 \
    --save_step 500

print("\n✅ GPT Training abgeschlossen!")

In [ ]:
# ============================================================
# ZELLE 7: DVAE Training (optional — nur bei >20h Audio)
# ============================================================
DVAE_TRAINING = False  #@param {type:"boolean"}
# False = überspringen (empfohlen für kurze Datasets <20h)
# True  = DVAE trainieren (nur bei >20 Stunden Audio sinnvoll)

if DVAE_TRAINING:
    print("🎛️ Starte DVAE Training (~2-3 Stunden)...")
    !python train_dvae.py \
        --dataset_path /content/dataset \
        --output_path /content/output_dvae \
        --batch_size 512 \
        --lr 5e-6 \
        --num_epochs 5
    print("✅ DVAE Training abgeschlossen")
else:
    print("⏭️ DVAE Training übersprungen (weniger als 20h Audio)")

In [ ]:
# ============================================================
# ZELLE 8: Modell exportieren & zu BookVoice hochladen
# ============================================================
# ⚠️ Erst ausführen wenn Training (Zelle 6) fertig ist!
import glob, shutil, zipfile

print("🔍 Suche trainiertes Modell...")

model_dir = Path("/content/final_model")
model_dir.mkdir(exist_ok=True)

# GPT Checkpoint
gpt_files = sorted(glob.glob("/content/output_gpt/**/*.pth", recursive=True))
if not gpt_files:
    gpt_files = sorted(glob.glob("/content/output_gpt/*.pth"))

if gpt_files:
    shutil.copy(gpt_files[-1], model_dir / "model.pth")
    print(f"✅ GPT Modell: {Path(gpt_files[-1]).name}")
else:
    raise Exception("❌ Kein GPT Checkpoint gefunden — Training abgeschlossen?")

# Config + Vocab vom Basis-Modell
for fname in ["config.json", "vocab.json"]:
    src = f"/content/xtts-training/{fname}"
    if Path(src).exists():
        shutil.copy(src, model_dir / fname)
        print(f"✅ {fname} kopiert")

# DVAE falls vorhanden
dvae_files = sorted(glob.glob("/content/output_dvae/**/*.pth", recursive=True))
if dvae_files:
    shutil.copy(dvae_files[-1], model_dir / "dvae.pth")
    print("✅ DVAE Checkpoint kopiert")

# ZIP erstellen
ZIP_PATH = f"/content/{MODELL_NAME}.zip"
print(f"\n📦 Erstelle ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in model_dir.rglob("*"):
        if f.is_file():
            z.write(f, f.name)

zip_size = Path(ZIP_PATH).stat().st_size / 1024 / 1024
print(f"✅ ZIP: {zip_size:.1f} MB")

# Hochladen
print(f"\n📤 Lade hoch zu BookVoice...")
with open(ZIP_PATH, 'rb') as f:
    r = requests.post(
        f"{API_BASE}/training/models/upload",
        data={"name": MODELL_NAME},
        files={"file": (f"{MODELL_NAME}.zip", f, "application/zip")},
        timeout=300
    )

if r.status_code == 200:
    d = r.json()
    print(f"\n✅ Modell hochgeladen!")
    print(f"   Name:    {d.get('name')}")
    print(f"   Dateien: {', '.join(d.get('dateien', []))}")
    print()
    print("🎉 Fertig! Jetzt in BookVoice:")
    print("   1. Trainingsraum öffnen")
    print(f"   2. Karte '4 · Trainiertes Modell'")
    print(f"   3. '{d.get('name')}' → Aktivieren")
    print("   4. Studio → Hörbuch mit eigener Stimme generieren! 🎙️")
else:
    print(f"❌ Upload fehlgeschlagen: {r.status_code}")
    print(r.text[:300])
    print("\n📥 Manueller Download:")
    from google.colab import files
    files.download(ZIP_PATH)

---
## 📝 Tipps & Hinweise

### Style-Tags
| Tag | Verwendung |
|---|---|
| `<sufi>text</sufi>` | Spirituelle, ruhige Texte |
| `<whisper>text</whisper>` | Flüsternde, geheimnisvolle Texte |
| `<dramatic>text</dramatic>` | Dramatische, intensive Texte |
| `<poem>text</poem>` | Gedichte, kurze Verse |

### Parameter-Empfehlungen
| Dataset | Epochs | Dauer |
|---|---|---|
| <5 Min Audio | 5-10 | ~1-2 Std |
| 5-15 Min Audio | 5 | ~2-4 Std |
| >20 Min Audio | 3-5 + DVAE | ~4-6 Std |

### Wichtig
- Immer **fp32** Training (`mixed_precision no`) — fp16 verursacht NaN Loss bei XTTS!
- **Python 3.10** verwenden — nicht 3.11 oder 3.12
- Colab-Tab **offen lassen** während des Trainings